# 01 · How much predictive power is actually there?

Before building anything, four questions, all answerable without a model:

1. **What does the illumination pattern look like, and what do the readouts do around it?**
2. **At what spatial scale** does the mask carry information — i.e. how far does a
   model need to be able to see?
3. **How much of the readout is predictable from space at all?** Much of the
   cell-to-cell variation is single-cell noise that no spatial model can reach.
4. **What is the density confound?** If illumination also changes cell density,
   then "predicted from BMP4" and "predicted from colony structure" are entangled.

The answers set every design choice in notebooks 02–04, so this notebook is the one
to re-run first if the data changes.

In [ ]:
import sys, os
sys.path.append('../src')

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tools.dataset import load_well, marker_positive, IF_BACKGROUNDS, PRIMARY_WELL, MATCHED_PAIR, RESULTS
from tools.spatial import PYRAMID_SCALES
import tools.evaluation as ev
import plotting as pl

pd.set_option('display.width', 160)
plt.rcParams['figure.dpi'] = 110

## Load

The binary `BMP4_bin` mask is the model input throughout — not the raw intensity.
Positive/negative calls use the thresholds verified by eye (see `tools.dataset`).

Note the field is loaded **uncropped**. The crop previously in use kept only the
illuminated ring and discarded the second pattern (a filled square at x ≈ 4700–5250),
i.e. half the experiment.

In [ ]:
df = load_well(PRIMARY_WELL)

df['Sox17_pos'] = marker_positive(df, 'Sox17')
df['T_pos'] = marker_positive(df, 'T')

centroids = df[['centroid_y', 'centroid_x']].to_numpy(np.float32)
print(f'{len(df):,} cells   field {centroids[:,0].max():.0f} x {centroids[:,1].max():.0f} px')
print(f"BMP4+ : {df['BMP4_bin'].mean():.1%}")
print(f"Sox17+: {df['Sox17_pos'].mean():.1%}   (threshold {IF_BACKGROUNDS['Sox17_mean']:.1f})")
print(f"T+    : {df['T_pos'].mean():.1%}   (threshold {IF_BACKGROUNDS['T_mean']:.1f})")

## 1 · What does it look like?

In [ ]:
fig = pl.plot_cell_maps(df, [
    ('BMP4+ (the model input)', 'BMP4_bin', 'gray_r'),
    ('Sox17 (log)', np.log1p(df['Sox17_mean']), 'magma'),
    ('T (log)', np.log1p(df['T_mean']), 'magma'),
    ('Hoechst (log)', np.log1p(df['Hoechst_mean']), 'viridis'),
], suptitle='per-cell maps')
plt.show()

At ~100k cells the per-cell scatter is too dense to read. The smoothed fields
below are the single most useful picture in this notebook — and the **cell density**
row is included deliberately, because a marker field and the density field are very
easy to confuse by eye.

In [ ]:
fig = pl.plot_field_grid(df, rows=[
    ('BMP4+', 'BMP4_bin'),
    ('Sox17+', 'Sox17_pos'),
    ('T+', 'T_pos'),
    ('cell density', None),
], sigmas=[30, 60, 120, 240])
plt.show()

**What to look for.** The mask is an illuminated ring (radius ≈ 450px) plus
scattered isolated positives. T is strongly suppressed in a disc that fills *and
slightly exceeds* the ring. Sox17 shows the same suppression far more weakly, and is
dominated by patchy colony-scale structure unrelated to the illumination.

Crucially: **cell density is also low inside the ring**. Keep that in view — notebook
02 quantifies it and notebook 04 turns it into a control.

## 2 · Is there a spatial relationship at all?\n\nThe most generous possible test: correlate the *smoothed fields*, so per-cell noise is averaged away. A near-zero value here means there is genuinely nothing to find, not that a model was too weak.

In [ ]:
from tools.spatial import smoothed_field

shape = (int(centroids[:,0].max())+1, int(centroids[:,1].max())+1)
rows = []
for sig in [30, 60, 120, 240, 480]:
    fields = {n: smoothed_field(centroids, df[c].to_numpy(np.float32), sig, shape_px=shape)
              for n, c in [('BMP4', 'BMP4_bin'), ('Sox17', 'Sox17_pos'), ('T', 'T_pos')]}
    dens = smoothed_field(centroids, np.ones(len(df), np.float32), sig, shape_px=shape) * 0
    from tools.spatial import rasterize
    from scipy.ndimage import gaussian_filter
    g, _, _ = rasterize(centroids, np.ones(len(df), np.float32), shape_px=shape)
    fields['density'] = gaussian_filter(g, sigma=sig/8.0, mode='nearest')
    for a, b in [('BMP4', 'Sox17'), ('BMP4', 'T'), ('density', 'Sox17'), ('density', 'T')]:
        m = np.isfinite(fields[a]) & np.isfinite(fields[b])
        rows.append(dict(sigma=sig, pair=f'{a} ~ {b}', r=float(np.corrcoef(fields[a][m], fields[b][m])[0,1])))

field_corr = pd.DataFrame(rows)
pl.plot_field_correlation(rows)
plt.show()
field_corr.pivot_table(index='sigma', columns='pair', values='r').round(3)

`BMP4 ~ T` reaches about **−0.76**: strong and negative, i.e. T is suppressed where
BMP4 is on. `BMP4 ~ Sox17` peaks around **−0.25** only.

But `density ~ T` reaches **+0.69** — nearly as strong as the BMP4 relationship, and
with the opposite sign. That is the confound, visible before any model exists.

## 3 · At what scale — and can the model even see that far?

In [ ]:
from tools.spatial import add_mask_pyramid

mask_cols = add_mask_pyramid(df, 'BMP4_bin', geometry=True)
print('added:', mask_cols)

pl.plot_scale_auroc(df, 'BMP4_bin_gauss_', PYRAMID_SCALES,
                    targets=[('Sox17+', df['Sox17_pos']), ('T+', df['T_pos'])])
plt.show()

The scale at which each curve departs from 0.5 is the length scale of the biology,
and it is what the graph radius has to be able to reach.

Now the blunt version of the same question — a cell with **no** BMP4+ cell anywhere
in its receptive field has a literally constant input, so nothing downstream can
distinguish it from any other such cell.

In [ ]:
pl.plot_receptive_field_check(df, 'BMP4_bin', radii=[30, 60, 90, 120, 240, 480, 960])
plt.axvline(90, color='r', ls='--', lw=1)
plt.gca().annotate('radius=30px x 3 layers', xy=(90, 0.5), xytext=(120, 0.62),
                   arrowprops=dict(arrowstyle='->', color='r'), color='r', fontsize=9)
plt.show()

**This is the headline design constraint.** A 30px radius with 3 message-passing
layers reaches ~90px, and the overwhelming majority of cells have no BMP4+ cell in
that range. Reaching ~1000px by stacking 30px hops would need ~30 layers; building an
explicit 960px-radius graph would need ~8×10⁸ edges. The workable route is a fixed
Gaussian **pyramid** of the mask as extra node features (`tools.spatial`) — long-range
context at O(pixels) per scale, with message passing still supplying fine structure.

## 4 · Response vs pattern geometry\n\nSigned distance to the illuminated boundary (+ inside, − outside) is the dose-response coordinate. It also distinguishes *inside a ring with no BMP4+ cell nearby* from *far outside the pattern with no BMP4+ cell nearby* — two situations every density feature reports identically, and the two the readouts differ most between.

In [ ]:
pl.plot_response_curve(df['BMP4_bin_signed_dist'], targets=[
    ('P(Sox17+)', df['Sox17_pos']), ('P(T+)', df['T_pos'])])
plt.show()

centre = pl.estimate_pattern_centre(df, 'BMP4_bin')
print('pattern centre (y, x) =', tuple(round(c) for c in centre))
pl.plot_radial_profile(df, centre, series=[
    ('BMP4+ fraction', 'BMP4_bin'), ('Sox17+ fraction', 'Sox17_pos'), ('T+ fraction', 'T_pos')])
plt.show()

## 5 · The ceiling — how much is predictable at all?

Predict each cell from its **neighbours' true values**, leave-one-out. That is allowed
to peek at the answer everywhere except the cell being scored, so it is an upper bound
on any spatial model. Whatever fraction of the per-cell variation is independent
single-cell noise is unreachable, and this measures exactly that.

Every score in notebooks 02–04 should be read as a fraction of this, **not** of 1.0.

In [ ]:
ceilings = {}
for marker in ['Sox17', 'T']:
    raw = df[f'{marker}_mean'].to_numpy(np.float32)
    y, ypos = ev.hurdle_target(raw, IF_BACKGROUNDS[f'{marker}_mean'])
    best = max((ev.oracle_ceiling(centroids, y, ypos, sigma=s) for s in [30, 60, 120]),
               key=lambda d: d['auroc'])
    ceilings[marker] = best
    print(f"{marker:6s} oracle ceiling:  R2 = {best['r2']:.3f}   AUROC = {best['auroc']:.3f}   (sigma={best['sigma']:g}px)")

pd.to_pickle(ceilings, f'{RESULTS}/oracle_ceilings.pkl')
print(f'\nsaved -> {RESULTS}/oracle_ceilings.pkl')

## What this implies

| finding | consequence |
|---|---|
| 86% of cells have no BMP4+ neighbour within 30px | a ~90px receptive field cannot work — **add multi-scale context features** |
| BMP4↔T field correlation −0.76; BMP4↔Sox17 only −0.25 | expect a good T model and a weak Sox17 one |
| density↔T correlation +0.69, and the ring is low-density | **density is a confound** — quantified in 02, controlled in 04 |
| Sox17 ceiling R² 0.25, T ceiling R² 0.44 | judge models against these, not against 1.0 |

Next: **02** measures how strongly the mask predicts density; **03** predicts the
markers from the mask; **04** repeats with density only, as the control.